In [1]:
# === IMPORTS ===
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dataclasses import dataclass
from typing import Tuple, Union, List, Dict, Optional
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, MaxAbsScaler
from sklearn.model_selection import KFold
from category_encoders import CatBoostEncoder  # Добавляем CatBoost Encoder



from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor, HuberRegressor
from sklearn.model_selection import cross_validate, GridSearchCV, StratifiedKFold
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge, Lasso, SGDRegressor, ElasticNet, HuberRegressor
from sklearn.model_selection import cross_validate, GridSearchCV, KFold
from sklearn.preprocessing import RobustScaler, OneHotEncoder, StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, 
    r2_score, mean_absolute_percentage_error, median_absolute_error,
    confusion_matrix, classification_report,
)
from scipy.stats import gmean
from scipy.optimize import minimize_scalar
from sklearn.compose import TransformedTargetRegressor
from typing import Dict, Tuple, List, Optional, Union
from dataclasses import dataclass

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Настройка визуализации
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✅ Все библиотеки успешно импортированы")

@dataclass
class Config:
    """Централизованная конфигурация проекта"""
    
    # Пути к файлам
    TRAIN_PATH: str = "train.csv"
    TEST_PATH: str = "test.csv"
    OUTPUT_PATH: str = "val_predictions.csv"
    
    # Параметры предобработки
    COLUMNS_TO_DROP: List[str] = None
    OUTLIER_COLUMNS: List[str] = None
    
    # Параметры модели
    CV_FOLDS: int = 5
    RANDOM_STATE: int = None
    N_JOBS: int = -1
    
    # Параметры сегментации
    SEGMENT_STEP: int = 5000
    MIN_PRICE = 0
    MAX_PRICE = 1000000
    BIN_SIZE = 5000

    encoding_type: str = "mean"   # "mean" | "target" | "ohe"
    te_n_splits: int = 5
    te_smoothing: float = 10.0    # сглаживание: чем больше, тем сильнее тянет к глобальному среднему
    te_random_state: int = 42

    #coords
    lat_center: float = 42.8637
    lon_center: float = 74.6057

    # Метрики для оптимизации
    PRIMARY_METRIC: str = "neg_mean_absolute_error" #neg_mean_absolute_percentage_error
    SCORING_METRICS: List[str] = None
    
    def __post_init__(self):
        if self.COLUMNS_TO_DROP is None:
            self.COLUMNS_TO_DROP = [
                #'Возможность рассрочки', 'Возможность обмена', 'Возможность ипотеки',
                'Площадь участка', "Канализация", "Питьевая вода",
                "Электричество",
            ]
        
        if self.OUTLIER_COLUMNS is None:
            #self.OUTLIER_COLUMNS = ['usd_price', 'area']
            #self.OUTLIER_COLUMNS = ['usd_price', 'added_days', 'area']
            #self.OUTLIER_COLUMNS = ['added_days']
            self.OUTLIER_COLUMNS = ['price_per_sqm']

        if self.SCORING_METRICS is None:
            self.SCORING_METRICS = [
                "r2", "neg_mean_absolute_error", 
                "neg_root_mean_squared_error", "neg_mean_absolute_percentage_error", "neg_mean_squared_error"
            ]

# Инициализация конфигурации
config = Config()
print("✅ Конфигурация проекта инициализирована")
print(f"📊 Основные параметры:")
print(f"   - Метрика оптимизации: {config.PRIMARY_METRIC}")
print(f"   - Количество фолдов CV: {config.CV_FOLDS}")
print(f"   - Шаг сегментации: ${config.SEGMENT_STEP:,}")
print(f"   - Диапазон цен: ${config.MIN_PRICE:,} - ${config.MAX_PRICE:,}")

✅ Все библиотеки успешно импортированы
✅ Конфигурация проекта инициализирована
📊 Основные параметры:
   - Метрика оптимизации: neg_mean_absolute_error
   - Количество фолдов CV: 5
   - Шаг сегментации: $5,000
   - Диапазон цен: $0 - $1,000,000


In [2]:
from pyexpat import features


class DataProcessor:
    """Класс для комплексной обработки данных с поддержкой нескольких типов кодировок категориальных признаков."""

    # ---- RU -> EN маппинг колонок
    RU2EN_MAP: Dict[str, str] = {
        # базовые
        "Тип предложения": "deal_type",
        "Серия": "series",
        "Дом": "house",
        "Этаж": "floor",
        "Площадь": "area_str",
        "Отопление": "heating",
        "Состояние": "status",
        "Газ": "gas",
        "Санузел": "bathroom",
        "Балкон": "balcony",
        "Входная дверь": "front_door",
        "Парковка": "parking",
        "Высота потолков": "ceiling_height",
        "Безопасность": "security",
        "Правоустанавливающие документы": "documents",
        "Телефон": "phone",
        "Интернет": "internet",
        "Мебель": "furniture",
        "Материал": "material",
        "Год постройки": "year_built",
        "Пол": "cover",
        "Площадь участка": "land_area",
        "Разное": "misc",
        "Канализация": "sewerage",
        "Питьевая вода": "drinking_water",
        "Электричество": "electricity",

        # твои флаги-удаляемые
        "Возможность рассрочки": "installment",
        "Возможность обмена": "exchange",
        "Возможность ипотеки": "mortgage",
    }

    def __init__(self, config: Config):
        self.config = config

        # глобальные значения для замены в новых категориях
        self.price_median: Optional[float] = 1400.0
        self.price_mean: Optional[float] = 1400.0
        self.price_gmean: Optional[float] = 1400.0

        # энкодеры/скейлер
        self.scaler = MaxAbsScaler()

        self.classifier = DecisionTreeClassifier(
            max_depth=10,
            min_samples_split=25,
            min_samples_leaf=10,
            random_state=config.RANDOM_STATE
        )

        self.cols = None
        self.best_model_choser = None
        self.best_model = None

        # для mean/target encoding
        self.encoding_maps_mean: Dict[str, Dict[str, Dict[str, float]]] = {}
        # теперь внутри каждой колонки храним три мапы: median/mean/gmean
        # пример: {"address_1": {"median": {...}, "mean": {...}, "gmean": {...}}, ...}

        self.te_global_mean: Optional[float] = None
        self.te_cat_maps: Dict[str, Dict[str, float]] = {}

        # для времени
        self.current_year: int = 2025

        print("✅ DataProcessor инициализирован")

    # =========================
    # Загрузка
    # =========================
    def load_data(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        try:
            train_df = pd.read_csv(self.config.TRAIN_PATH)
            test_df = pd.read_csv(self.config.TEST_PATH)

            print(f"✅ Данные загружены:")
            print(f"   - Train: {train_df.shape}")
            print(f"   - Test:  {test_df.shape}")
            return train_df, test_df
        except Exception as e:
            print(f"❌ Ошибка загрузки данных: {e}")
            raise

    # =========================
    # Базовые утилиты
    # =========================
    def rename_to_english(self, df: pd.DataFrame) -> pd.DataFrame:
        """Переименовывает колонки на английский (выполняется ПЕРВЫМ шагом)."""
        return df.rename(columns=self.RU2EN_MAP)

    def _drop_known_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Удаляет мусорные/ненужные колонки. Поддерживает ru/en названия из конфига."""
        # полный список к удалению: как в конфиге, так и их английские аналоги
        drop_candidates = set(self.config.COLUMNS_TO_DROP)
        # добавить канд. на английском, если есть в словаре
        for ru in self.config.COLUMNS_TO_DROP:
            if ru in self.RU2EN_MAP:
                drop_candidates.add(self.RU2EN_MAP[ru])

        drop_list = ['id']
        drop_candidates.update(drop_list)

        existing = [c for c in drop_candidates if c in df.columns]
        if existing:
            df = df.drop(columns=existing)
            print(f"🧹 Удалено {len(existing)} колонок: {existing}")
        return df

    # =========================
    # Пропуски
    # =========================
    def fill_missing_values_mode(self, df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
        """Заполняет пропуски модой для перечисленных колонок (и безопасно дропает остатки, если is_train)."""
        df = df.copy()

        # Множество колонок, где логично заполнять модой
        fill_mode_cols = {
            'address', 'deal_type', 'series', 'heating', 'status', 'gas',
       'bathroom', 'balcony', 'front_door', 'parking', 'ceiling_height',
       'security', 'documents', 'phone', 'internet', 'furniture', 'cover',
       'address_1',
       'address_2', 'material', 'view_count', 'hearts', 'lat', 'lon', 'floor',
       'usd_price', 'added_days', 'upped_days', 'year_built', 'rooms', 'area',
       'area_per_room', 'max_floor', 'floor_ratio', 'age', 'price_per_sqm',
       'price_per_room', 'log_area', 'sqrt_area', 'area_sq'
        }

        filled = 0
        for col in fill_mode_cols:
            if col in df.columns:
                miss = df[col].isna().sum()
                if miss > 0:
                    mode_val = df[col].mode(dropna=True)
                    if not mode_val.empty:
                        df[col] = df[col].fillna(mode_val.iloc[0])
                        filled += miss

        if filled:
            print(f"🔧 Заполнено {filled} пропусков модой")

        if is_train:
            before = len(df)
            df = df.dropna()
            removed = before - len(df)
            if removed:
                print(f"🔧 Удалено {removed} строк с оставшимися пропусками (train)")
        return df

    def fill_missing_values(self, df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
        """Заполняет пропуски значением NA для перечисленных колонок (и безопасно дропает остатки, если is_train)."""
        df = df.copy()

        # Множество колонок, где логично заполнять NA
        fill_na_cols = df.select_dtypes(include=['object']).columns

        filled = 0
        for col in fill_na_cols:
            if col in df.columns:
                miss = df[col].isna().sum()
                if miss > 0:
                    df[col] = df[col].fillna("NA")
                    filled += miss

        fill_median_cols = df.select_dtypes(include=['number']).columns

        for col in fill_median_cols:
            if col in df.columns:
                miss = df[col].isna().sum()
                if miss > 0:
                    df[col] = df[col].fillna(df[col].median())
                    filled += miss

        if filled:
            print(f"🔧 Заполнено {filled} пропусков значением NA")

        if is_train:
            before = len(df)
            df = df.dropna()
            removed = before - len(df)
            if removed:
                print(f"🔧 Удалено {removed} строк с оставшимися пропусками (train)")

        return df

    def embedding_sentence_transformer(self, df: pd.DataFrame) -> pd.DataFrame:
        """Создает эмбеддинги для текстовых колонок с помощью SentenceTransformer."""
        from sentence_transformers import SentenceTransformer

        text_cols = df.select_dtypes(include=['object']).columns
        model = SentenceTransformer("google/embeddinggemma-300m", device='cuda')

        from sklearn.decomposition import TruncatedSVD

        # ...existing code...
        df['all_text'] = df[text_cols].astype(str).agg(' '.join, axis=1)
        df_embeddings = model.encode(df['all_text'], show_progress_bar=True)

        # SVD до 32 компонент
        svd = TruncatedSVD(n_components=16, random_state=42)
        svd_features = svd.fit_transform(df_embeddings)

        # Добавляем к df новые колонки
        for i in range(16):
            df[f'text_svd_{i+1}'] = svd_features[:, i]
        # ...existing code...

        df = df.drop(columns=['all_text'])
        print(f"📝 Созданы текстовые эмбеддинги с помощью SentenceTransformer и SVD")
        
        return df
    
    def similarity_to_class(self, df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
        # Используем только числовые признаки
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        # Исключаем целевые и служебные столбцы, если они есть
        for col in ['usd_price', 'real_price_per_sqm', 'price_per_sqm', 'price_class']:
            if col in num_cols:
                num_cols.remove(col)

        if is_train:
            if 'usd_price' in df.columns and 'area' in df.columns:
                df['price_per_sqm'] = df['usd_price'] / df['area']
            if 'price_per_sqm' in df.columns:
                q1 = df['price_per_sqm'].quantile(0.01)
                q50 = df['price_per_sqm'].quantile(0.50)
                q99 = df['price_per_sqm'].quantile(0.99)
                # Классы: 1 — <=1%, 2 — (1%, 50%], 3 — (50%, 99%], 4 — >99%
                def price_class(x):
                    if x <= q1:
                        return '1'
                    elif x <= q50:
                        return '2'
                    elif x <= q99:
                        return '3'
                    else:
                        return '4'
                df['price_class'] = df['price_per_sqm'].map(price_class)

            cheap_mask = df['price_class'] == '1'
            normal_mask = df['price_class'] == '2'
            expensive_mask = df['price_class'] == '3'
            ultra_mask = df['price_class'] == '4'

            df_scaled = self.scaler.fit_transform(df[num_cols])

            self.cheap_mean = df_scaled[cheap_mask].mean(axis=0) if cheap_mask.any() else np.zeros(len(num_cols))
            self.normal_mean = df_scaled[normal_mask].mean(axis=0) if normal_mask.any() else np.zeros(len(num_cols))
            self.expensive_mean = df_scaled[expensive_mask].mean(axis=0) if expensive_mask.any() else np.zeros(len(num_cols))
            self.ultra_mean = df_scaled[ultra_mask].mean(axis=0) if ultra_mask.any() else np.zeros(len(num_cols))

            df = df.drop(columns=['price_class', 'price_per_sqm'])

        def cosine_similarity(X, Y):
            X = np.asarray(X)
            Y = np.asarray(Y)
            if Y.ndim == 1:
                Y = Y.reshape(1, -1)
            X_norm = np.linalg.norm(X, axis=1, keepdims=True)
            Y_norm = np.linalg.norm(Y, axis=1, keepdims=True)
            sim = np.dot(X, Y.T) / (X_norm * Y_norm.T + 1e-8)
            return sim

        X_num = self.scaler.transform(df[num_cols])
        df['sim_cheap'] = cosine_similarity(X_num, self.cheap_mean).flatten()
        df['sim_normal'] = cosine_similarity(X_num, self.normal_mean).flatten()
        df['sim_expensive'] = cosine_similarity(X_num, self.expensive_mean).flatten()
        df['sim_ultra'] = cosine_similarity(X_num, self.ultra_mean).flatten()
        return df
        

    # =========================
    # Фичи
    # =========================
    @staticmethod
    def _parse_added_or_upped(text: str) -> Optional[float]:
        """
        Преобразует строки вида:
          - 'Добавлено 7 дней назад'
          - 'Поднято 8 часов назад'
          - 'Добавлено 2 месяца назад'
        в число дней (float). Только split/простейшие проверки.
        """
        if not isinstance(text, str):
            return None
        parts = text.split()
        if len(parts) < 3:
            return None

        # ожидаем число на второй позиции
        try:
            value = int(parts[1])
        except Exception:
            return None

        unit = parts[2].lower()

        # очень простые эвристики по подстрокам
        if 'мин' in unit:      # минуты
            return value / (24 * 60)
        if 'час' in unit or 'ч' == unit:   # часы
            return value / 24
        if 'дн' in unit or unit.startswith('д'):
            return float(value)
        if 'месяц' in unit or 'ме' in unit:
            return float(value) * 30.0
        if 'год' in unit or unit.startswith('г'):
            return float(value) * 365.0
        return None

    def _process_added_time(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'added' in df.columns:
            s = df['added'].apply(self._parse_added_or_upped)
            df['added_days'] = s.fillna(s.median())
            df = df.drop(columns=['added'])
        return df

    def _process_upped_time(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'upped' in df.columns:
            s = df['upped'].apply(self._parse_added_or_upped)
            df['upped_days'] = s.fillna(s.median())
            df = df.drop(columns=['upped'])
        return df

    def _process_hearts(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'hearts' in df.columns:
            df['hearts'] = pd.to_numeric(df['hearts'], errors='coerce').fillna(0).astype(int)
        return df

    def _process_address(self, df: pd.DataFrame) -> pd.DataFrame:
        """Адрес разбиваем на 2 части, максимально безопасно."""
        if 'address' in df.columns:
            def part(x: str, idx: int):
                if isinstance(x, str):
                    parts = [p.strip() for p in x.split(',')]
                    if len(parts) > idx:
                        return parts[idx]
                return x
            df['address_1'] = df['address'].map(lambda x: part(x, 1))
            df['address_2'] = df['address'].map(lambda x: part(x, 2))
        return df.drop(columns=['address'])

    def _process_coords(self, train: pd.DataFrame, test: pd.DataFrame) -> tuple:
        """
        Обрабатывает координаты: добавляет кластеризацию KMeans и расстояние до центра.
        Объединяет train и test для корректного обучения кластеризатора.
        Возвращает обработанные train и test.
        """
        from sklearn.cluster import KMeans

        # Объединяем для fit
        all_df = pd.concat([train, test], axis=0, ignore_index=True)
        coords = all_df[['lat', 'lon']].fillna(0)

        # Центр по всем данным
        self.lat_center_computed = coords['lat'].median()
        self.lon_center_computed = coords['lon'].median()
        print(f"📍 Центр определен автоматически: ({self.lat_center_computed:.4f}, {self.lon_center_computed:.4f})")

        # Кластеризация по всем валидным координатам
        valid_coords = coords[(coords['lat'] != 0) & (coords['lon'] != 0)]
        n_clusters = 20
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        self.kmeans.fit(valid_coords)

        # Присваиваем кластеры
        cluster_labels = np.full(len(all_df), -1)
        mask = (coords['lat'] != 0) & (coords['lon'] != 0)
        cluster_labels[mask] = self.kmeans.predict(coords[mask])
        all_df['geo_cluster'] = cluster_labels.astype(str)

        # Расчет расстояния до центра
        def haversine_distance(lat1, lon1, lat2, lon2):
            R = 6371
            lat1_rad = np.radians(lat1)
            lat2_rad = np.radians(lat2)
            delta_lat = np.radians(lat2 - lat1)
            delta_lon = np.radians(lon2 - lon1)
            a = np.sin(delta_lat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(delta_lon/2)**2
            c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
            return R * c

        all_df['distance_from_center'] = haversine_distance(
            all_df['lat'],
            all_df['lon'],
            self.lat_center_computed,
            self.lon_center_computed
        )

        all_df = all_df.drop(columns=['lat', 'lon'])

        # Разделяем обратно
        train_out = all_df.iloc[:len(train)].reset_index(drop=True)
        test_out = all_df.iloc[len(train):].reset_index(drop=True)
        return train_out, test_out

    def _process_building_info(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'house' in df.columns:
            def house_part(x: str, idx: int):
                if isinstance(x, str):
                    parts = [p.strip() for p in x.split(',')]
                    return parts[idx] if len(parts) > idx else np.nan
                return np.nan
            df['material'] = df.get('material', df['house'].map(lambda x: house_part(x, 0)))
            df['year_built'] = df.get('year_built', df['house'].map(lambda x: house_part(x, 1)))
        return df

    def _process_main_info(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'main' in df.columns:
            # rooms
            def rooms_from_main(x: str):
                if isinstance(x, str) and len(x) > 0:
                    if x[0] == 'с':  # студия
                        return 7
                    if x[0].isdigit():
                        return int(x[0])
                return np.nan
            # area
            def area_from_main(x: str):
                # ожидаем "3, 75 м²" и т.п.
                if isinstance(x, str) and ', ' in x:
                    try:
                        return float(x.split(', ')[1].split()[0])
                    except Exception:
                        return np.nan
                return np.nan

            df['rooms'] = df['main'].map(rooms_from_main)
            df['area'] = df['main'].map(area_from_main)
            # если area уже выделили — площадь на комнату
            df['area_per_room'] = df['area'] / df['rooms']
            df['room_per_area'] = df['rooms'] / df['area']

        return df

    def _process_floor_info(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'floor' in df.columns:
            # max_floor
            def max_floor(x: str):
                if isinstance(x, str):
                    toks = x.split()
                    if toks and toks[-1].isdigit():
                        return int(toks[-1])
                return np.nan

            # current floor
            def cur_floor(x: str):
                if isinstance(x, str):
                    first = x.split()[0]
                    if first in ['цоколь', 'подвал']:
                        return 0
                    if first.isdigit():
                        return int(first)
                return np.nan

            df['max_floor'] = df['floor'].map(max_floor)
            default_max = df['max_floor'].mode().iloc[0] if not df['max_floor'].mode().empty else 5
            df['max_floor'] = df['max_floor'].fillna(default_max)

            df['floor'] = df['floor'].map(cur_floor)
            df['floor_ratio'] = df['floor'].replace(0, 1) / df['max_floor']
        return df

    def _process_year_and_age(self, df: pd.DataFrame) -> pd.DataFrame:
        if 'year_built' in df.columns:
            def year_num(x):
                if isinstance(x, str):
                    tok = x.split()[0]
                    return int(tok) if tok.isdigit() else np.nan
                return x
            df['year_built'] = df['year_built'].map(year_num)
            default_year = df['year_built'].mode().iloc[0] if not df['year_built'].mode().empty else 2000
            df['year_built'] = df['year_built'].fillna(default_year)

            df['age'] = self.current_year - df['year_built']
            default_age = df['age'].mode().iloc[0] if not df['age'].mode().empty else 25
            df['age'] = df['age'].fillna(default_age)
            q05 = df['age'].quantile(0.05)
            q25 = df['age'].quantile(0.25)
            q75 = df['age'].quantile(0.75)
            q95 = df['age'].quantile(0.95)
            def age_cat(x):
                if x < q05:
                    return 'super_new'
                elif x < q25:
                    return 'new'
                elif x > q95:
                    return 'super_old'
                elif x > q75:
                    return 'old'
                else:
                    return 'mid'
            #df['age'] = df['age'].map(age_cat)

        return df.drop(columns='year_built')
    
    def _proccess_ceiling_height(self, train: pd.DataFrame, test: pd.DataFrame) -> tuple:
        """
        Обрабатывает высоту потолков: объединяет train и test, приводит к float, заполняет пропуски и 0 модой.
        Возвращает обработанные train и test.
        """
        # Объединяем для единой обработки
        all_df = pd.concat([train, test], axis=0, ignore_index=True)

        # Преобразуем к float, 'NA' → 2.5
        all_df['ceiling_height'] = all_df['ceiling_height'].map(lambda x: float(x.split()[0]) if x != 'NA' else 2.5)

        # Заполняем пропуски модой
        mode_val = all_df['ceiling_height'].mode()[0]
        all_df['ceiling_height'] = all_df['ceiling_height'].fillna(mode_val)
        # Заменяем 0 на моду
        all_df['ceiling_height'] = all_df['ceiling_height'].replace(0, mode_val)

        # Разделяем обратно
        train_out = all_df.iloc[:len(train)].reset_index(drop=True)
        test_out = all_df.iloc[len(train):].reset_index(drop=True)
        return train_out, test_out
    
    def _process_documents(self, train: pd.DataFrame, test: pd.DataFrame) -> tuple:
        """
        Создаёт бинарные признаки по всем уникальным видам документов.
        Объединяет train и test для единого набора уникальных значений.
        """
        all_df = pd.concat([train, test], axis=0, ignore_index=True)
        docs_series = all_df['documents'].astype(str).str.lower()
        unique_docs = set()
        for doc_list in docs_series:
            if doc_list != 'na':
                unique_docs.update([d.strip() for d in doc_list.split(',') if d.strip()])

        for doc in unique_docs:
            col_name = f"doc_{doc.replace(' ', '_')}"
            all_df[col_name] = docs_series.apply(lambda x: int(doc in x))

        all_df['documents'] = all_df['documents'].map(lambda x: 0 if x == 'NA' else len([d for d in x.split(',') if d.strip()]))

        train_out = all_df.iloc[:len(train)].reset_index(drop=True)
        test_out = all_df.iloc[len(train):].reset_index(drop=True)
        return train_out, test_out

    def _process_security(self, train: pd.DataFrame, test: pd.DataFrame) -> tuple:
        """
        Создаёт бинарные признаки по всем уникальным видам безопасности.
        Объединяет train и test для единого набора уникальных значений.
        """
        all_df = pd.concat([train, test], axis=0, ignore_index=True)
        sec_series = all_df['security'].astype(str).str.lower()
        unique_sec = set()
        for sec_list in sec_series:
            if sec_list != 'na':
                unique_sec.update([s.strip() for s in sec_list.split(',') if s.strip()])

        for sec in unique_sec:
            col_name = f"sec_{sec.replace(' ', '_')}"
            all_df[col_name] = sec_series.apply(lambda x: int(sec in x))

        all_df['security'] = all_df['security'].map(lambda x: 0 if x == 'NA' else len([s for s in x.split(',') if s.strip()]))

        train_out = all_df.iloc[:len(train)].reset_index(drop=True)
        test_out = all_df.iloc[len(train):].reset_index(drop=True)
        return train_out, test_out

    def _process_misc(self, train: pd.DataFrame, test: pd.DataFrame) -> tuple:
        """
        Создаёт бинарные признаки по всем уникальным видам удобств.
        Объединяет train и test для единого набора уникальных значений.
        """
        all_df = pd.concat([train, test], axis=0, ignore_index=True)
        misc_series = all_df['misc'].astype(str).str.lower()
        unique_misc = set()
        for misc_list in misc_series:
            if misc_list != 'na':
                unique_misc.update([m.strip() for m in misc_list.split(',') if m.strip()])

        for misc in unique_misc:
            col_name = f"misc_{misc.replace(' ', '_')}"
            all_df[col_name] = misc_series.apply(lambda x: int(misc in x))

        all_df['misc'] = all_df['misc'].map(lambda x: 0 if x == 'NA' else len([m for m in x.split(',') if m.strip()]))

        train_out = all_df.iloc[:len(train)].reset_index(drop=True)
        test_out = all_df.iloc[len(train):].reset_index(drop=True)
        return train_out, test_out

    def _cleanup_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Удаляет временные колонки."""
        cols_to_remove = [c for c in ['main', 'area_str', 'house'] if c in df.columns]
        if cols_to_remove:
            df = df.drop(columns=cols_to_remove)
        return df
    
    def _encode_deal_type(self, df: pd.DataFrame) -> pd.DataFrame:
        df['deal_type'] = df['deal_type'].map(lambda x: 1 if x == 'от агента' else 0)
        return df

    def extract_features(self, train: pd.DataFrame, test: pd.DataFrame) -> tuple:
        """Полный набор фич-инжиниринга для train и test одновременно."""
        tr, te = train.copy(), test.copy()

        # Методы, которые требуют объединения train+test
        tr, te = self._process_coords(tr, te)
        tr, te = self._proccess_ceiling_height(tr, te)
        tr, te = self._process_documents(tr, te)
        tr, te = self._process_security(tr, te)
        tr, te = self._process_misc(tr, te)

        # Методы, которые можно применять по отдельности
        for fn in [
            self._process_added_time,
            self._process_upped_time,
            self._process_hearts,
            self._process_address,
            self._process_building_info,
            self._process_main_info,
            self._process_floor_info,
            self._process_year_and_age,
            self._cleanup_columns,
            self._encode_deal_type
        ]:
            tr = fn(tr)
            te = fn(te)
        return tr, te

    # =========================
    # Удаление выбросов
    # =========================
    def remove_outliers(self, df: pd.DataFrame, start: float = 0.25, end: float = 0.75) -> pd.DataFrame:
        """Стандартный IQR-фильтр по колонкам из конфига."""
        cleaned = df.copy()
        cleaned['price_per_sqm'] = df['usd_price'] / df['area']
        total_removed = 0
        for col in self.config.OUTLIER_COLUMNS:
            if col in cleaned.columns:
                before = len(cleaned)
                Q1 = cleaned[col].quantile(start)
                Q3 = cleaned[col].quantile(end)
                IQR = Q3 - Q1
                lower = Q1 - 1.5 * IQR
                upper = Q3 + 1.5 * IQR
                cleaned = cleaned[(cleaned[col] >= lower) & (cleaned[col] <= upper)]
                total_removed += (before - len(cleaned))
        if total_removed:
            print(f"🧹 Удалено {total_removed} выбросов (IQR) от {start} до {end}")
        return cleaned.drop(columns=['price_per_sqm']).reset_index(drop=True)
    
    def remove_by_cut(self, df: pd.DataFrame, start: float = 0.25, end: float = 0.75) -> pd.DataFrame:
        """Стандартный IQR-фильтр по колонкам из конфига."""
        cleaned = df.copy()
        total_removed = 0
        for col in self.config.OUTLIER_COLUMNS:
            if col in cleaned.columns:
                before = len(cleaned)
                Q1 = cleaned[col].quantile(start)
                Q3 = cleaned[col].quantile(end)
                cleaned = cleaned[(cleaned[col] >= Q1) & (cleaned[col] <= Q3)]
                total_removed += (before - len(cleaned))
        if total_removed:
            print(f"🧹 Удалено {total_removed} выбросов от {start} до {end}")
        return cleaned

    # =========================
    # Кодирование категориальных
    # =========================

    def _encode_label(self, train: pd.DataFrame, test: pd.DataFrame) -> tuple:
        """Label Encoding категориальных признаков: fit на train+test, transform по отдельности."""
        from sklearn.preprocessing import LabelEncoder

        tr = train.copy()
        te = test.copy()
        all_df = pd.concat([tr, te], axis=0, ignore_index=True)
        cat_columns = all_df.select_dtypes(include=['object']).columns.tolist()
        if not cat_columns:
            print("⚠️ Категориальных признаков не найдено")
            return tr, te

        self.label_encoders = {}
        for col in cat_columns:
            le = LabelEncoder()
            le.fit(all_df[col].astype(str))
            tr[col] = tr[col].map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
            te[col] = te[col].map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
            self.label_encoders[col] = le

        print(f"🔤 (Label) Закодировано {len(cat_columns)} категориальных признаков")
        print(f"   Признаки: {cat_columns}")
        return tr, te
    
    def _encode_catboost(self, df: pd.DataFrame, 
                         is_train: bool = True) -> pd.DataFrame:
        """CatBoost Encoding категориальных признаков"""
        X = df.copy()

        # Определяем категориальные колонки
        if is_train:
            target = df['usd_price'] / df['area']
            self.cat_columns = X.select_dtypes(include=['object']).columns.tolist()
            
            if not self.cat_columns:
                print("⚠️ Категориальных признаков не найдено")
                return X
            
            if target is None:
                raise ValueError("Для обучения CatBoost Encoder необходим target")
            
            # Создаем и обучаем CatBoost Encoder
            self.catboost_encoder = CatBoostEncoder(cols=self.cat_columns, random_state=self.config.RANDOM_STATE)
            X_encoded = self.catboost_encoder.fit_transform(X.drop(columns='usd_price'), target)
            X_encoded['usd_price'] = df['usd_price']
            print(f"🔤 (CatBoost) Закодировано {len(self.cat_columns)} категориальных признаков")
            print(f"   Признаки: {self.cat_columns}")
            
        else:
            if self.catboost_encoder is None:
                raise RuntimeError("CatBoost Encoder не обучен. Сначала вызовите с is_train=True")
            
            # Применяем обученный энкодер
            X_encoded = self.catboost_encoder.transform(X)
            print(f"🔤 (CatBoost) Применено кодирование к тестовым данным")
        
        return X_encoded
    
    def _encode_mean(self, df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
        """Mean/Median/Geometric encoding категориальных признаков"""
        features = pd.DataFrame(index=df.index)

        # Базовые значения по умолчанию
        if is_train:
            df['real_price_per_sqm'] = df['usd_price'] / df['area']
            self.price_median = float(df['real_price_per_sqm'].median())
            self.price_mean = float(df['real_price_per_sqm'].mean())
            self.price_gmean = float(gmean(df['real_price_per_sqm'][df['real_price_per_sqm'] > 0]))  # только положительные
        if not hasattr(self, "price_median"):
            self.price_median = 1400.0
            self.price_mean = 1400.0
            self.price_gmean = 1400.0

        cat_cols = df.select_dtypes(include=['object']).columns

        for col in cat_cols:
            if is_train and 'real_price_per_sqm' in df.columns:
                stats = df.groupby(col)['real_price_per_sqm'].agg(['median', 'mean'])
                
                # геометрическое среднее — только на положительных
                gmeans = df[df['real_price_per_sqm'] > 0].groupby(col)['real_price_per_sqm'].apply(gmean)
                
                features[f"{col}__median"] = df[col].map(stats['median'])
                features[f"{col}__mean"]   = df[col].map(stats['mean'])
                features[f"{col}__gmean"]  = df[col].map(gmeans)
                features[f"{col}"]  = (features[f"{col}__median"] + features[f"{col}__mean"] + features[f"{col}__gmean"]) / 3
                features.drop(columns=[f"{col}__median", f"{col}__mean", f"{col}__gmean"], inplace=True)

                # сохраняем карты
                self.encoding_maps_mean[col] = {
                    "median": stats['median'].to_dict(),
                    "mean": stats['mean'].to_dict(),
                    "gmean": gmeans.to_dict()
                }
            else:
                mapping = self.encoding_maps_mean.get(col, {})
                features[f"{col}__median"] = df[col].map(mapping.get("median", {})).fillna(self.price_median)
                features[f"{col}__mean"]   = df[col].map(mapping.get("mean", {})).fillna(self.price_mean)
                features[f"{col}__gmean"]  = df[col].map(mapping.get("gmean", {})).fillna(self.price_gmean)
                features[f"{col}"]  = (features[f"{col}__median"] + features[f"{col}__mean"] + features[f"{col}__gmean"]) / 3
                features.drop(columns=[f"{col}__median", f"{col}__mean", f"{col}__gmean"], inplace=True)

        # Добавляем числовые колонки как есть
        num_cols = df.select_dtypes(include=['number', 'bool']).columns
        features[num_cols] = df[num_cols]

        def weighted_address(row):
            c1 = row['address_1_count']
            c2 = row['address_2_count']
            # Добавим +1 чтобы избежать деления на 0
            w1 = 1 / (c1 + 1)
            w2 = 1 / (c2 + 1)
            # Нормируем веса
            total = w1 + w2
            w1 /= total
            w2 /= total
            return w1 * row['address_1'] + w2 * row['address_2']

        features['address_weighted'] = features.apply(weighted_address, axis=1)

        features['addresses_mean'] = (features['address_1'] + features['address_2'])/2

        print(f"🔤 (mean+median+gmean) Закодировано {len(cat_cols)} категориальных признаков × 3")
        return features

    def _encode_ohe(self, df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
        """One-Hot Encoding с динамическим drop:
        - если в колонке есть 'NA' → дропаем 'NA'
        - иначе дропаем моду (конкретную категорию), а не 'first'
        Храним список колонок и drop_map с fit, корректно работаем на test.
        """
        X = df.copy()

        cat_cols = X.select_dtypes(include=['object']).columns.tolist()
        cat_cols = [c for c in cat_cols if c in X.columns and c not in ['address_1', 'address_2', 'geo_cluster']]
        if not cat_cols:
            # Нечего кодировать — просто вернуть X как есть
            return X

        # Приведём категориальные к строкам (на случай числовых кодов) и добьём пропуски
        X[cat_cols] = X[cat_cols].astype(str).fillna("NA")

        if is_train:
            # Сформировать корректный drop_map (только конкретные категории, без "first")
            drop_map = {}
            for col in cat_cols:
                uniques = pd.Series(X[col].unique())
                if (uniques == "NA").any():
                    drop_map[col] = "NA"
                else:
                    m = X[col].mode(dropna=True)
                    drop_map[col] = m.iloc[0] if not m.empty else (uniques.iloc[0] if len(uniques) else "NA")

            # Создать и обучить OHE
            self.ohe = OneHotEncoder(sparse_output=False, drop="if_binary", handle_unknown="ignore")
            self.ohe.fit(X[cat_cols])

            # Сохранить служебную инфу для test
            self.ohe_fitted = True
            self.ohe_cats = cat_cols
            self.ohe_drop_map = drop_map

        else:
            # Должны уже быть обучены
            if not getattr(self, "ohe_fitted", False):
                raise RuntimeError("OneHotEncoder не обучен. Сначала вызови _encode_ohe(..., is_train=True).")

            # Гарантируем наличие всех обученных колонок в тесте
            missing = [c for c in self.ohe_cats if c not in X.columns]
            for col in missing:
                X[col] = "NA"
            # Строго порядок обученных колонок
            cat_cols = list(self.ohe_cats)

        # Трансформация
        ohe_arr = self.ohe.transform(X[cat_cols])
        ohe_cols = self.ohe.get_feature_names_out(cat_cols)
        ohe_df = pd.DataFrame(ohe_arr, columns=ohe_cols, index=X.index)
        #ohe_df = ohe_df.drop(columns=[i for i in ohe_cols if i.split('_')[-1] in ['NA', 
        #                                                                          #'нет', 'пустая', 'не достроено', 'малосемейка'
        #                                                                          ]])
        # Присоединяем прочие (не-категориальные) колонки
        other_cols = X.columns.difference(cat_cols)
        out = pd.concat([X[other_cols], ohe_df], axis=1)

        print(f"🔠 (ohe) Закодировано {len(cat_cols)} категориальных признаков → {ohe_df.shape[1]} столбцов")
        return out
    
    # =========================
    # Доп. признаки от площади
    # =========================
    def process_area_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Генерирует дополнительные числовые признаки вокруг площади. Ожидает, что вход уже числовой (после encoding)."""
        X = df.copy()
        if 'area' in X.columns:
            #if 'rooms' in X.columns:
                #X['area_per_room'] = X['area'] / X['rooms'].replace(0, 1)
                #X['area_per_room'] = X['area_per_room'].map(lambda x: 'high' if x > X['area_per_room'].quantile(0.75) else ('low' if x < X['area_per_room'].quantile(0.25) else 'mid'))

            # простые нелинейные преобразования
            X['log_area'] = np.log1p(X['area'])
            X['sqrt_area'] = np.sqrt(np.clip(X['area'], a_min=0, a_max=None))
            X['area_sq'] = X['area'] ** 2
            X['volume'] = X['area'] * X['ceiling_height']
            #X['area_price'] = X[['address_1', 'address_2', 'series', 'status', 'ceiling_height', 'furniture', 'material']].mean(axis=1) * X['area']
            #X['area_price_2'] = X[['deal_type', 'heating', 'gas', 'front_door', 'parking', 'furniture', 'security', 'documents']].mean(axis=1) * X['area']
        return X
    
    def process_address_stats(self, train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
        """Создаёт статистические признаки по адресам."""
        X = train.copy()
        TEST = test.copy()

        intersect_cols = set(X.columns).intersection(set(TEST.columns))
        TRAIN = X[list(intersect_cols)]
        TEST = TEST[list(intersect_cols)]

        DF = pd.concat([TRAIN, TEST], axis=0).reset_index(drop=True)
        # Количество объявлений по address_1
        addr1_counts = DF['address_1'].value_counts().to_dict()
        self.address_1_counts = addr1_counts

        # Количество объявлений по address_2
        addr2_counts = DF['address_2'].value_counts().to_dict()
        self.address_2_counts = addr2_counts

        # Средний год постройки по address_1
        addr1_year_stats = DF.groupby('address_1')['age'].mean()
        self.address_1_year_stats = addr1_year_stats.to_dict()
        addr2_year_stats = DF.groupby('address_2')['age'].mean()
        self.address_2_year_stats = addr2_year_stats.to_dict()

        # Средняя максимальная этажность по address_1
        addr1_max_floor_stats = DF.groupby('address_1')['max_floor'].mean()
        self.address_1_max_floor_stats = addr1_max_floor_stats.to_dict()

        addr2_max_floor_stats = DF.groupby('address_2')['max_floor'].mean()
        self.address_2_max_floor_stats = addr2_max_floor_stats.to_dict()

        # Среднее количество просмотров по address_1
        addr1_view_stats = DF.groupby('address_1')['view_count'].mean()
        self.address_1_view_stats = addr1_view_stats.to_dict()
        addr2_view_stats = DF.groupby('address_2')['view_count'].mean()
        self.address_2_view_stats = addr2_view_stats.to_dict()

        # Среднее количество лайков по address_1
        addr1_hearts_stats = DF.groupby('address_1')['hearts'].mean()
        self.address_1_hearts_stats = addr1_hearts_stats.to_dict()
        addr2_hearts_stats = DF.groupby('address_2')['hearts'].mean()
        self.address_2_hearts_stats = addr2_hearts_stats.to_dict()

        # Средняя дата добавления и поднятия по address_1
        addr1_added_days_stats = DF.groupby('address_1')['added_days'].mean()
        self.address_1_added_days_stats = addr1_added_days_stats.to_dict()
        addr2_added_days_stats = DF.groupby('address_2')['added_days'].mean()
        self.address_2_added_days_stats = addr2_added_days_stats.to_dict()
        addr1_upped_days_stats = DF.groupby('address_1')['upped_days'].mean()
        self.address_1_upped_days_stats = addr1_upped_days_stats.to_dict()
        addr2_upped_days_stats = DF.groupby('address_2')['upped_days'].mean()
        self.address_2_upped_days_stats = addr2_upped_days_stats.to_dict()
        
        # Применение статистик
        X['address_1_count'] = TRAIN['address_1'].map(addr1_counts).fillna(0)
        X['address_1_avg_age'] = TRAIN['address_1'].map(addr1_year_stats).fillna(TRAIN['age'].median())
        X['address_2_count'] = TRAIN['address_2'].map(getattr(self, 'address_2_counts', {})).fillna(0)
        X['address_2_avg_age'] = TRAIN['address_2'].map(getattr(self, 'address_2_year_stats', {})).fillna(TRAIN['age'].median())
        X['address_1_avg_max_floor'] = TRAIN['address_1'].map(getattr(self, 'address_1_max_floor_stats', {})).fillna(TRAIN['max_floor'].median())
        X['address_2_avg_max_floor'] = TRAIN['address_2'].map(getattr(self, 'address_2_max_floor_stats', {})).fillna(TRAIN['max_floor'].median())
        X['address_1_avg_view_count'] = TRAIN['address_1'].map(getattr(self, 'address_1_view_stats', {})).fillna(TRAIN['view_count'].median())
        X['address_2_avg_view_count'] = TRAIN['address_2'].map(getattr(self, 'address_2_view_stats', {})).fillna(TRAIN['view_count'].median())
        X['address_1_avg_hearts'] = TRAIN['address_1'].map(getattr(self, 'address_1_hearts_stats', {})).fillna(TRAIN['hearts'].median())
        X['address_2_avg_hearts'] = TRAIN['address_2'].map(getattr(self, 'address_2_hearts_stats', {})).fillna(TRAIN['hearts'].median())
        X['address_1_avg_added_days'] = TRAIN['address_1'].map(getattr(self, 'address_1_added_days_stats', {})).fillna(TRAIN['added_days'].median())
        X['address_2_avg_added_days'] = TRAIN['address_2'].map(getattr(self, 'address_2_added_days_stats', {})).fillna(TRAIN['added_days'].median())
        X['address_1_avg_upped_days'] = TRAIN['address_1'].map(getattr(self, 'address_1_upped_days_stats', {})).fillna(TRAIN['upped_days'].median())
        X['address_2_avg_upped_days'] = TRAIN['address_2'].map(getattr(self, 'address_2_upped_days_stats', {})).fillna(TRAIN['upped_days'].median())

        TEST['address_1_count'] = TEST['address_1'].map(getattr(self, 'address_1_counts', {})).fillna(0)
        TEST['address_1_avg_age'] = TEST['address_1'].map(getattr(self, 'address_1_year_stats', {})).fillna(TEST['age'].median())
        TEST['address_2_count'] = TEST['address_2'].map(getattr(self, 'address_2_counts', {})).fillna(0)
        TEST['address_2_avg_age'] = TEST['address_2'].map(getattr(self, 'address_2_year_stats', {})).fillna(TEST['age'].median())
        TEST['address_1_avg_max_floor'] = TEST['address_1'].map(getattr(self, 'address_1_max_floor_stats', {})).fillna(TEST['max_floor'].median())
        TEST['address_2_avg_max_floor'] = TEST['address_2'].map(getattr(self, 'address_2_max_floor_stats', {})).fillna(TEST['max_floor'].median())
        TEST['address_1_avg_view_count'] = TEST['address_1'].map(getattr(self, 'address_1_view_stats', {})).fillna(TEST['view_count'].median())
        TEST['address_2_avg_view_count'] = TEST['address_2'].map(getattr(self, 'address_2_view_stats', {})).fillna(TEST['view_count'].median())
        TEST['address_1_avg_hearts'] = TEST['address_1'].map(getattr(self, 'address_1_hearts_stats', {})).fillna(TEST['hearts'].median())
        TEST['address_2_avg_hearts'] = TEST['address_2'].map(getattr(self, 'address_2_hearts_stats', {})).fillna(TEST['hearts'].median())
        TEST['address_1_avg_added_days'] = TEST['address_1'].map(getattr(self, 'address_1_added_days_stats', {})).fillna(TEST['added_days'].median())
        TEST['address_2_avg_added_days'] = TEST['address_2'].map(getattr(self, 'address_2_added_days_stats', {})).fillna(TEST['added_days'].median())
        TEST['address_1_avg_upped_days'] = TEST['address_1'].map(getattr(self, 'address_1_upped_days_stats', {})).fillna(TEST['upped_days'].median())
        TEST['address_2_avg_upped_days'] = TEST['address_2'].map(getattr(self, 'address_2_upped_days_stats', {})).fillna(TEST['upped_days'].median())
        
        return X, TEST
    # =========================
    # Список фич и масштабирование
    # =========================
    def prepare_features(self, df: pd.DataFrame) -> List[str]:
        """Определяет список признаков для модели (фильтр по существующим)."""
        base = [
            # категориальные (после кодирования остаются числовые)
            'deal_type', 'series', 'heating', 'status', 'gas',
            'front_door', 'parking', 'ceiling_height', 'security', 'documents',
            'furniture', 'material',
            # числовые
            'floor', 'rooms', 'area', 'max_floor', 'age',
            # поведенческие/временные
            'added_days', 'upped_days', 'hearts', 'view_count',
            # таргет-производные
            #'price_per_sqm', 'price_per_room',
            
            # area-derived
            'area_per_room', 'log_area', 'sqrt_area', 'area_sq',
            # адресные (после кодирования станут числовые)
            'address_1', 'address_2', 'area_price_2', 'area_price',
        ]
        return [c for c in base if c in df.columns]

    # =========================
    # Главный пайплайн
    # =========================
    def process_pipeline(self, train_df: pd.DataFrame, test_df: pd.DataFrame, remove_outliers: bool = True,
                    start: float = 0.25, end: float = 0.75, encode: str = 'target') -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Полный пайплайн:
        1) RENAME → 2) DROP → 3) FILL → 4) FEATURES → 5) PRICE → 6) OUTLIERS (опц.) → 7) ENCODE → 8) AREA-FEATS
        Возвращает: train_encoded, test_encoded
        """
        print("🔄 Запуск пайплайна обработки данных...")

        # ---------- TRAIN ----------
        print("\n📊 Обработка TRAIN:")
        tr = train_df.copy()
        tr = self.rename_to_english(tr)           # (1)
        tr = self._drop_known_columns(tr)         # (2)
        tr = self.fill_missing_values(tr, is_train=True)  # (3)

        # ---------- TEST ----------
        print("\n📋 Обработка TEST:")
        te = test_df.copy()
        te = self.rename_to_english(te)
        te = self._drop_known_columns(te)
        te = self.fill_missing_values(te, is_train=False)

        # ---------- FEATURES (совместно) ----------
        tr, te = self.extract_features(tr, te)
        tr = self.process_area_features(tr)
        te = self.process_area_features(te)

        tr, te = self.process_address_stats(tr, te)

        if remove_outliers:
            tr = self.remove_outliers(tr, start=start, end=end)  # (6)

        if encode == 'target':
            tr_enc = self._encode_ohe(tr, is_train=True)
            tr_enc = self._encode_mean(tr_enc, is_train=True)    # (7)

            te_enc = self._encode_ohe(te, is_train=False)
            te_enc = self._encode_mean(te_enc, is_train=False)    # (7)
            
        elif encode == 'catboost':
            tr_enc = self._encode_catboost(tr, is_train=True)
            te_enc = self._encode_catboost(te, is_train=False)

        elif encode == 'label':
            tr_enc = self._encode_ohe(tr, is_train=True)
            te_enc = self._encode_ohe(te, is_train=False)
            tr_enc, te_enc = self._encode_label(tr_enc, te_enc)

        tr_enc = tr_enc.dropna()  # на всякий случай
        tr_enc = tr_enc.reset_index(drop=True)

        print(f"\n✅ Пайплайн завершен:\n   - Train: {tr_enc.shape}\n   - Test:  {te_enc.shape}")
        return tr_enc, te_enc

processor = DataProcessor(config)

✅ DataProcessor инициализирован


In [3]:
class ResultsManager:
    """Класс для управления финальными результатами"""
    
    def __init__(self, config: Config):
        self.config = config
    
    def generate_predictions(self, model,
                           X_test_scaled: np.ndarray, test_df: pd.DataFrame,
                           is_per_sqm: bool) -> pd.DataFrame:
        """Генерирует финальные предсказания"""
        print("🎯 Генерация финальных предсказаний...")
        
        # Базовые предсказания
        test_pred_base = model.predict(X_test_scaled) * X_test_scaled['area'] if is_per_sqm else model.predict(X_test_scaled)
        
        # Создаем DataFrame с результатами
        submission = pd.DataFrame({
            'id': test_df['id'],
            'usd_price': test_pred_base
        })
        
        # Статистика предсказаний
        stats = {
            'count': len(test_pred_base),
            'min': test_pred_base.min(),
            'max': test_pred_base.max(),
            'mean': test_pred_base.mean(),
            'median': np.median(test_pred_base),
            'std': test_pred_base.std()
        }
        
        print("📊 Статистика финальных предсказаний:")
        print(f"   Количество: {stats['count']:,}")
        print(f"   Минимум: ${stats['min']:,.2f}")
        print(f"   Максимум: ${stats['max']:,.2f}")
        print(f"   Среднее: ${stats['mean']:,.2f}")
        print(f"   Медиана: ${stats['median']:,.2f}")
        print(f"   Стд. отклонение: ${stats['std']:,.2f}")
        
        return submission, stats
    
    def save_results(self, submission: pd.DataFrame, filename: str = None):
        """Сохраняет результаты в файл"""
        if filename is None:
            filename = self.config.OUTPUT_PATH
        
        submission.to_csv(filename, index=False)
        print(f"💾 Результаты сохранены в: {filename}")
        
        return filename
    
# Инициализация менеджера результатов
results_manager = ResultsManager(config)


In [4]:
# Загрузка данных
train_df, test_df = processor.load_data()

# Полная обработка данных
raw_data, raw_test = processor.process_pipeline(train_df, test_df, True, start=0, end=0.99, encode='label')  # сырые данные с фичами, но без кодирования
#train_data, train_test = processor.process_pipeline(train_df, test_df, True, start=0, end=0.75)  # обучающая и тестовая выборки с фичами и кодированием

print("✅ Данные загружены и обработаны")

✅ Данные загружены:
   - Train: (7134, 36)
   - Test:  (1784, 36)
🔄 Запуск пайплайна обработки данных...

📊 Обработка TRAIN:
🧹 Удалено 4 колонок: ['sewerage', 'land_area', 'electricity', 'drinking_water']
🔧 Заполнено 82682 пропусков значением NA

📋 Обработка TEST:
🧹 Удалено 5 колонок: ['sewerage', 'land_area', 'id', 'electricity', 'drinking_water']
🔧 Заполнено 20402 пропусков значением NA
📍 Центр определен автоматически: (42.8453, 74.6124)
🧹 Удалено 2 выбросов (IQR) от 0 до 0.99
🔠 (ohe) Закодировано 16 категориальных признаков → 89 столбцов
🔠 (ohe) Закодировано 16 категориальных признаков → 89 столбцов
🔤 (Label) Закодировано 3 категориальных признаков
   Признаки: ['address_1', 'address_2', 'geo_cluster']

✅ Пайплайн завершен:
   - Train: (7128, 156)
   - Test:  (1784, 156)
✅ Данные загружены и обработаны


In [5]:
raw_test = raw_test.drop(columns=['usd_price'], errors='ignore')
cols_list = raw_test.columns.tolist()


X = raw_data[cols_list].copy()
y = raw_data['usd_price'] / raw_data['area']

X_test = raw_test

In [6]:
X_test

,added_days,address_1,address_1_avg_added_days,address_1_avg_age,address_1_avg_hearts,address_1_avg_max_floor,address_1_avg_upped_days,address_1_avg_view_count,address_1_count,address_2,address_2_avg_added_days,address_2_avg_age,address_2_avg_hearts,address_2_avg_max_floor,address_2_avg_upped_days,address_2_avg_view_count,address_2_count,age,area,area_per_room,area_sq,ceiling_height,deal_type,distance_from_center,doc_акт_ввода_в_эксплуатацию,doc_договор_дарения,doc_договор_долевого_участия,doc_договор_купли-продажи,doc_зеленая_книга,doc_красная_книга,doc_свидетельство_о_праве_на_наследство,doc_технический_паспорт,documents,floor,floor_ratio,geo_cluster,hearts,log_area,max_floor,misc,misc_встроенная_кухня,misc_кладовка,misc_комнаты_изолированы,misc_кондиционер,misc_кухня-студия,misc_неугловая,misc_новая_сантехника,misc_пластиковые_окна,misc_тихий_двор,misc_удобно_под_бизнес,misc_улучшенная,room_per_area,rooms,sec_видеодомофон,sec_видеонаблюдение,sec_домофон,sec_кодовый_замок,sec_консьерж,sec_охрана,sec_решетки_на_окнах,sec_сигнализация,security,sqrt_area,upped_days,view_count,volume,series_104 серия,series_104 серия улучшенная,series_105 серия,series_105 серия улучшенная,series_106 серия,series_106 серия улучшенная,series_107 серия,series_108 серия,series_NA,series_индивид. планировка,series_малосемейка,series_пентхаус,series_сталинка,series_хрущевка,series_элитка,heating_NA,heating_автономное,heating_без отопления,heating_на газе,heating_на твердом топливе,heating_смешанное,heating_центральное,heating_электрическое,status_NA,status_евроремонт,status_не достроено,status_под самоотделку (псо),status_среднее,status_хорошее,gas_NA,gas_автономный,gas_возможно подключение,gas_магистральный,gas_нет,bathroom_2 с/у и более,bathroom_NA,bathroom_нет,bathroom_раздельный,bathroom_совмещенный,balcony_NA,balcony_балкон,balcony_застекленный балкон,balcony_лоджия,balcony_нет,front_door_NA,front_door_бронированная,front_door_деревянная,front_door_металлическая,front_door_нет,parking_NA,parking_гараж,parking_паркинг,parking_рядом охраняемая стоянка,phone_NA,phone_возможно подключение,phone_есть,phone_нет,internet_NA,internet_adsl,internet_оптика,internet_проводной,furniture_NA,furniture_полностью меблирована,furniture_пустая,furniture_частично меблирована,installment_NA,installment_есть,installment_нет,exchange_NA,exchange_ключ на ключ,exchange_обмен на авто,exchange_обмен не предлагать,exchange_рассмотрю варианты,exchange_с доплатой покупателя,exchange_с доплатой продавца,cover_NA,cover_дерево,cover_ковролин,cover_ламинат,cover_линолеум,cover_паркет,cover_плитка,cover_пробковое,mortgage_NA,mortgage_есть,mortgage_нет,material_кирпичный,material_монолитный,material_панельный
0,19.0,350,58.847879,2.389091,3.276364,11.014545,3.584091,457.225455,275,1064,40.000000,1.000000,1.500000,9.750000,0.447917,246.750000,4,1.0,46.50,46.500000,2162.2500,2.8,1,4.736906,0,0,0,0,0,0,0,0,0,11,0.916667,6,2,3.860730,12.0,0,0,0,0,0,0,0,0,0,0,0,0,0.021505,1,0,0,0,0,0,0,0,0,0,6.819091,0.375000,112,130.200,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,13.0,12,68.497234,7.754386,3.099415,10.526316,2.028996,484.479532,171,14,40.333333,4.000000,2.000000,10.000000,7.625000,341.833333,6,5.0,48.00,24.000000,2304.0000,3.0,1,4.283744,0,0,0,1,0,1,0,1,3,9,0.900000,6,1,3.891820,10.0,4,0,0,0,0,1,1,0,1,1,0,0,0.041667,2,0,1,1,1,0,1,0,0,4,6.928203,0.375000,176,144.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [7]:
X

,added_days,address_1,address_1_avg_added_days,address_1_avg_age,address_1_avg_hearts,address_1_avg_max_floor,address_1_avg_upped_days,address_1_avg_view_count,address_1_count,address_2,address_2_avg_added_days,address_2_avg_age,address_2_avg_hearts,address_2_avg_max_floor,address_2_avg_upped_days,address_2_avg_view_count,address_2_count,age,area,area_per_room,area_sq,ceiling_height,deal_type,distance_from_center,doc_акт_ввода_в_эксплуатацию,doc_договор_дарения,doc_договор_долевого_участия,doc_договор_купли-продажи,doc_зеленая_книга,doc_красная_книга,doc_свидетельство_о_праве_на_наследство,doc_технический_паспорт,documents,floor,floor_ratio,geo_cluster,hearts,log_area,max_floor,misc,misc_встроенная_кухня,misc_кладовка,misc_комнаты_изолированы,misc_кондиционер,misc_кухня-студия,misc_неугловая,misc_новая_сантехника,misc_пластиковые_окна,misc_тихий_двор,misc_удобно_под_бизнес,misc_улучшенная,room_per_area,rooms,sec_видеодомофон,sec_видеонаблюдение,sec_домофон,sec_кодовый_замок,sec_консьерж,sec_охрана,sec_решетки_на_окнах,sec_сигнализация,security,sqrt_area,upped_days,view_count,volume,series_104 серия,series_104 серия улучшенная,series_105 серия,series_105 серия улучшенная,series_106 серия,series_106 серия улучшенная,series_107 серия,series_108 серия,series_NA,series_индивид. планировка,series_малосемейка,series_пентхаус,series_сталинка,series_хрущевка,series_элитка,heating_NA,heating_автономное,heating_без отопления,heating_на газе,heating_на твердом топливе,heating_смешанное,heating_центральное,heating_электрическое,status_NA,status_евроремонт,status_не достроено,status_под самоотделку (псо),status_среднее,status_хорошее,gas_NA,gas_автономный,gas_возможно подключение,gas_магистральный,gas_нет,bathroom_2 с/у и более,bathroom_NA,bathroom_нет,bathroom_раздельный,bathroom_совмещенный,balcony_NA,balcony_балкон,balcony_застекленный балкон,balcony_лоджия,balcony_нет,front_door_NA,front_door_бронированная,front_door_деревянная,front_door_металлическая,front_door_нет,parking_NA,parking_гараж,parking_паркинг,parking_рядом охраняемая стоянка,phone_NA,phone_возможно подключение,phone_есть,phone_нет,internet_NA,internet_adsl,internet_оптика,internet_проводной,furniture_NA,furniture_полностью меблирована,furniture_пустая,furniture_частично меблирована,installment_NA,installment_есть,installment_нет,exchange_NA,exchange_ключ на ключ,exchange_обмен на авто,exchange_обмен не предлагать,exchange_рассмотрю варианты,exchange_с доплатой покупателя,exchange_с доплатой продавца,cover_NA,cover_дерево,cover_ковролин,cover_ламинат,cover_линолеум,cover_паркет,cover_плитка,cover_пробковое,mortgage_NA,mortgage_есть,mortgage_нет,material_кирпичный,material_монолитный,material_панельный
0,60.0,250,101.586207,11.215517,4.353448,9.301724,2.468894,724.284483,116,1250,130.222222,31.444444,1.777778,7.888889,0.513889,629.777778,9,8.0,113.80,37.933333,12950.4400,3.0,1,3.774869,0,0,0,0,0,0,0,0,0,10.0,0.833333,0,1,4.743191,12.0,0,0,0,0,0,0,0,0,0,0,0,0,0.026362,3,0,0,0,0,0,0,0,0,0,10.667708,0.500000,302,341.400,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,7.0,350,58.847879,2.389091,3.276364,11.014545,3.584091,457.225455,275,2041,17.368056,2.000000,2.333333,12.000000,6.548611,157.333333,6,2.0,66.00,33.000000,4356.0000,3.0,1,4.619840,0,0,0,0,0,0,0,0,0,10.0,0.833333,6,2,4.204693,12.0,4,0,1,0,0,0,1,0,1,1,0,0,0.030303,2,1,1,1,1,0,1,0,0,4,8.124038,0.291667,102,198.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

In [8]:
cat_cols = [
    'address_1',
    'address_2',
    'geo_cluster',
    'deal_type',

    'doc_акт_ввода_в_эксплуатацию',
    'doc_договор_дарения',
    'doc_договор_долевого_участия',
    'doc_договор_купли-продажи',
    'doc_зеленая_книга',
    'doc_красная_книга',
    'doc_свидетельство_о_праве_на_наследство',
    'doc_технический_паспорт',

    'misc_встроенная_кухня',
    'misc_кладовка',
    'misc_комнаты_изолированы',
    'misc_кондиционер',
    'misc_кухня-студия',
    'misc_неугловая',
    'misc_новая_сантехника',
    'misc_пластиковые_окна',
    'misc_тихий_двор',
    'misc_удобно_под_бизнес',
    'misc_улучшенная',

    'sec_видеодомофон',
    'sec_видеонаблюдение',
    'sec_домофон',
    'sec_кодовый_замок',
    'sec_консьерж',
    'sec_охрана',
    'sec_решетки_на_окнах',
    'sec_сигнализация',

    'series_104 серия',
    'series_104 серия улучшенная',
    'series_105 серия',
    'series_105 серия улучшенная',
    'series_106 серия',
    'series_106 серия улучшенная',
    'series_107 серия',
    'series_108 серия',
    'series_NA',
    'series_индивид. планировка',
    'series_малосемейка',
    'series_пентхаус',
    'series_сталинка',
    'series_хрущевка',
    'series_элитка',

    'heating_NA',
    'heating_автономное',
    'heating_без отопления',
    'heating_на газе',
    'heating_на твердом топливе',
    'heating_смешанное',
    'heating_центральное',
    'heating_электрическое',

    'status_NA',
    'status_евроремонт',
    'status_не достроено',
    'status_под самоотделку (псо)',
    'status_среднее',
    'status_хорошее',

    'gas_NA',
    'gas_автономный',
    'gas_возможно подключение',
    'gas_магистральный',
    'gas_нет',

    'bathroom_2 с/у и более',
    'bathroom_NA',
    'bathroom_нет',
    'bathroom_раздельный',
    'bathroom_совмещенный',

    'balcony_NA',
    'balcony_балкон',
    'balcony_застекленный балкон',
    'balcony_лоджия',
    'balcony_нет',

    'front_door_NA',
    'front_door_бронированная',
    'front_door_деревянная',
    'front_door_металлическая',
    'front_door_нет',

    'parking_NA',
    'parking_гараж',
    'parking_паркинг',
    'parking_рядом охраняемая стоянка',

    'phone_NA',
    'phone_возможно подключение',
    'phone_есть',
    'phone_нет',

    'internet_NA',
    'internet_adsl',
    'internet_оптика',
    'internet_проводной',

    'furniture_NA',
    'furniture_полностью меблирована',
    'furniture_пустая',
    'furniture_частично меблирована',

    'installment_NA',
    'installment_есть',
    'installment_нет',

    'exchange_NA',
    'exchange_ключ на ключ',
    'exchange_обмен на авто',
    'exchange_обмен не предлагать',
    'exchange_рассмотрю варианты',
    'exchange_с доплатой покупателя',
    'exchange_с доплатой продавца',

    'cover_NA',
    'cover_дерево',
    'cover_ковролин',
    'cover_ламинат',
    'cover_линолеум',
    'cover_паркет',
    'cover_плитка',
    'cover_пробковое',

    'mortgage_NA',
    'mortgage_есть',
    'mortgage_нет',

    'material_кирпичный',
    'material_монолитный',
    'material_панельный'
]

num_cols = [c for c in X.columns if c not in cat_cols and c != "usd_price"]

In [9]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler

# =====================================================
# SPEED
# =====================================================
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")


# =====================================================
# SIMILARITIES
# =====================================================
def sim_x(q_cat, q_num, X_cat, X_num):
    cat_sim = (X_cat == q_cat.unsqueeze(0)).float()

    d = torch.abs(X_num - q_num.unsqueeze(0))
    scale = torch.abs(q_num.unsqueeze(0)) + 1e-6

    sim1 = 1.0 / (1.0 + d)
    sim2 = torch.exp(-d / scale)
    sim3 = torch.clamp(1.0 - d / scale, min=0.0)

    return torch.cat([cat_sim, sim1, sim2, sim3], dim=1)


def sim_xy(q_cat, q_num, q_y, X_cat, X_num, Y):
    Mx = sim_x(q_cat, q_num, X_cat, X_num)

    dy = torch.abs(Y - q_y)
    scale = torch.abs(q_y) + 1e-6

    y_sim1 = 1.0 / (1.0 + dy / scale)
    y_sim2 = torch.exp(-dy / scale)

    return torch.cat([Mx, y_sim1.unsqueeze(1), y_sim2.unsqueeze(1)], dim=1)


# =====================================================
# MODEL
# =====================================================
class ThreeLayerMoE(nn.Module):
    def __init__(self, d1, d2, n_samples, hidden=128, n_heads=8):
        super().__init__()

        self.n_heads = n_heads

        self.w1 = nn.Parameter(torch.ones(d1) * 0.1)
        self.w2 = nn.Parameter(torch.randn(n_heads, d2) * 0.1)

        # bias для каждого train объекта
        self.sample_bias = nn.Parameter(torch.zeros(n_samples))

        self.gating = nn.Sequential(
            nn.Linear(d2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_heads)
        )

        self.residual_mlps = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d2, hidden),
                nn.ReLU(),
                nn.Linear(hidden, 1)
            )
            for _ in range(n_heads)
        ])


# =====================================================
# TRAIN
# =====================================================
def train_model(
    df,
    Y,
    cat_cols,
    num_cols,
    epochs=20,
    lr=1e-2,
    min_lr=1e-6,
    lambda_res=0.05,
    lambda_entropy=0.05,
    lambda_div=0.05,
    g=0.5,
    device="cuda"
):
    X_cat = torch.tensor(df[cat_cols].values, dtype=torch.float32, device=device)
    X_num = torch.tensor(df[num_cols].values, dtype=torch.float32, device=device)
    y = torch.tensor(np.asarray(Y), dtype=torch.float32, device=device)

    n = len(df)

    d1 = sim_x(X_cat[0], X_num[0], X_cat, X_num).shape[1]
    d2 = sim_xy(
        X_cat[0],
        X_num[0],
        torch.tensor(0.0, device=device),
        X_cat,
        X_num,
        y
    ).shape[1]

    model = ThreeLayerMoE(
        d1=d1,
        d2=d2,
        n_samples=n
    ).to(device)

    model = torch.compile(model, mode="reduce-overhead")

    optimizer = optim.Adam(model.parameters(), lr=lr)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=min_lr
    )

    scaler = torch.amp.GradScaler(device=device)

    I = torch.eye(model.n_heads, device=device)

    # =====================================================
    for epoch in range(epochs):
        model.train()

        total_mape = 0.0

        pbar = tqdm(range(n), desc=f"Epoch {epoch+1}/{epochs}")

        for i in pbar:
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device):

                # ===== L1 =====
                M1 = sim_x(X_cat[i], X_num[i], X_cat, X_num)

                s1 = M1 @ model.w1
                s1[i] = -1e9

                a1 = torch.softmax(s1, dim=0)
                y1 = (a1 * y).sum()

                # ===== L2 =====
                M2 = sim_xy(
                    X_cat[i],
                    X_num[i],
                    y1,
                    X_cat,
                    X_num,
                    y
                )

                scores2 = M2 @ model.w2.T

                # bias по train объектам
                scores2 = scores2 + model.sample_bias[:, None]

                scores2[i] = -1e9

                a2 = torch.softmax(scores2, dim=0)

                y_heads = (a2 * y.unsqueeze(1)).sum(dim=0)

                feats_heads = (
                    a2.unsqueeze(2) * M2.unsqueeze(1)
                ).sum(dim=0)

                # ===== GATING =====
                feats_global = feats_heads.mean(dim=0)

                g_logits = model.gating(feats_global)
                g_weights = torch.softmax(g_logits, dim=0)

                # ===== RESIDUAL =====
                residual_heads = torch.stack([
                    model.residual_mlps[h](feats_heads[h])
                    for h in range(model.n_heads)
                ]).squeeze()

                # ===== SOFTMIN =====
                soft_weights = torch.softmax(
                    -torch.abs(residual_heads),
                    dim=0
                )

                # ===== COMBINE =====
                final_weights = (
                    g * g_weights
                    + (1 - g) * soft_weights
                )

                final_weights = (
                    final_weights
                    / (final_weights.sum() + 1e-8)
                )

                # ===== OUTPUT =====
                y2 = (final_weights * y_heads).sum()
                residual = (
                    final_weights * residual_heads
                ).sum()

                pred = y2 + residual
                true = y[i]

                # ===== LOSS =====
                mape = torch.abs(
                    (true - pred) / (true + 1e-6)
                )

                loss_reg = (
                    residual / (true + 1e-6)
                ).pow(2)

                entropy = -(
                    g_weights
                    * torch.log(g_weights + 1e-8)
                ).sum()

                # diversity
                W = model.w2

                W_norm = (
                    W
                    / (W.norm(dim=1, keepdim=True) + 1e-8)
                )

                sim = W_norm @ W_norm.T

                div_loss = (
                    sim - I
                ).pow(2).mean()

                loss = (
                    mape
                    + lambda_res * loss_reg
                    - lambda_entropy * entropy
                    + lambda_div * div_loss
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_mape += mape.item()

            pbar.set_postfix(
                MAPE=f"{100 * total_mape / (i + 1):.4f}%"
            )

        scheduler.step()

        print(
            f"Epoch {epoch+1}: "
            f"MAPE={100 * total_mape / n:.4f}%"
        )

    return model


# =====================================================
# PREDICT
# =====================================================
@torch.no_grad()
def predict(
    model,
    train_df,
    train_y,
    test_df,
    cat_cols,
    num_cols,
    g=0.5,
    device="cuda"
):
    model.eval()

    X_cat_tr = torch.tensor(
        train_df[cat_cols].values,
        dtype=torch.float32,
        device=device
    )

    X_num_tr = torch.tensor(
        train_df[num_cols].values,
        dtype=torch.float32,
        device=device
    )

    y_tr = torch.tensor(
        np.asarray(train_y),
        dtype=torch.float32,
        device=device
    )

    X_cat_te = torch.tensor(
        test_df[cat_cols].values,
        dtype=torch.float32,
        device=device
    )

    X_num_te = torch.tensor(
        test_df[num_cols].values,
        dtype=torch.float32,
        device=device
    )

    preds = []

    for i in tqdm(range(len(test_df)), desc="Predict"):

        # ===== L1 =====
        M1 = sim_x(
            X_cat_te[i],
            X_num_te[i],
            X_cat_tr,
            X_num_tr
        )

        s1 = M1 @ model.w1

        a1 = torch.softmax(s1, dim=0)
        y1 = (a1 * y_tr).sum()

        # ===== L2 =====
        M2 = sim_xy(
            X_cat_te[i],
            X_num_te[i],
            y1,
            X_cat_tr,
            X_num_tr,
            y_tr
        )

        scores2 = M2 @ model.w2.T

        # bias по train объектам
        scores2 = scores2 + model.sample_bias[:, None]

        a2 = torch.softmax(scores2, dim=0)

        y_heads = (
            a2 * y_tr.unsqueeze(1)
        ).sum(dim=0)

        feats_heads = (
            a2.unsqueeze(2)
            * M2.unsqueeze(1)
        ).sum(dim=0)

        # ===== GATING =====
        feats_global = feats_heads.mean(dim=0)

        g_weights = torch.softmax(
            model.gating(feats_global),
            dim=0
        )

        # ===== RESIDUAL =====
        residual_heads = torch.stack([
            model.residual_mlps[h](feats_heads[h])
            for h in range(model.n_heads)
        ]).squeeze()

        # ===== SOFTMIN =====
        soft_weights = torch.softmax(
            -torch.abs(residual_heads),
            dim=0
        )

        final_weights = (
            g * g_weights
            + (1 - g) * soft_weights
        )

        final_weights = (
            final_weights
            / (final_weights.sum() + 1e-8)
        )

        y2 = (final_weights * y_heads).sum()

        residual = (
            final_weights * residual_heads
        ).sum()

        pred = y2 + residual

        preds.append(pred.item())

    return np.array(preds)

In [10]:
t_model = train_model(
    X,
    y,
    cat_cols,
    num_cols,
    epochs=10,
    lr=1e-2,
    min_lr=1e-6,
    lambda_res=0.05,
    lambda_entropy=0.05,
    lambda_div=0.20,
    g=0.5,
    device="cpu"
)

Epoch 1/10:   0%|          | 0/7128 [00:00<?, ?it/s]

Epoch 1: MAPE=8.4146%


Epoch 2/10:   0%|          | 0/7128 [00:00<?, ?it/s]

Epoch 2: MAPE=7.5670%


Epoch 3/10:   0%|          | 0/7128 [00:00<?, ?it/s]

...

Epoch 8: MAPE=5.4169%


Epoch 9/10:   0%|          | 0/7128 [00:00<?, ?it/s]

Epoch 9: MAPE=5.2711%


Epoch 10/10:   0%|          | 0/7128 [00:00<?, ?it/s]

Epoch 10: MAPE=5.1645%


In [12]:
test_preds = predict(
    t_model,
    X,
    y,
    X_test,
    cat_cols,
    num_cols,
    g=0.5,
    device="cpu"
)
ids = test_df['id'].values
submission = pd.DataFrame({
    'id': ids,
    'usd_price': test_preds * X_test['area'].values  # если модель предсказывает price_per_sqm
})
submission.to_csv("three_layer_submission.csv", index=False)
print("✅ Предсказания сохранены в three_layer_submission.csv")

Predict:   0%|          | 0/1784 [00:00<?, ?it/s]

✅ Предсказания сохранены в three_layer_submission.csv
